# MNIST Digit Classifier

Train a convolutional neural network to recognize handwritten digits. **Runtime: set Runtime > Change runtime type > GPU.**

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 2. Load MNIST

In [ ]:
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_ds = datasets.MNIST('.', train=True, download=True, transform=tfm)
test_ds = datasets.MNIST('.', train=False, download=True, transform=tfm)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=256)
print(len(train_ds), 'training images')

## 3. Define a small CNN
Two conv layers + pooling, then a classifier head.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc = nn.Linear(32 * 7 * 7, 10)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        return self.fc(x.flatten(1))

model = Net().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

## 4. Train
Forward, loss, backward, step.

In [ ]:
for epoch in range(2):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = F.cross_entropy(model(xb), yb)
        loss.backward()
        opt.step()
    print(f'epoch {epoch + 1}: loss {loss.item():.3f}')

## 5. Evaluate

In [ ]:
model.eval()
correct = 0
with torch.no_grad():
    for xb, yb in test_dl:
        xb, yb = xb.to(device), yb.to(device)
        correct += (model(xb).argmax(1) == yb).sum().item()
print('test accuracy:', round(correct / len(test_ds), 4))

## Homework
1. Plot a few test images with predictions.
2. Find and view the images it gets wrong.
3. Add dropout or a third conv layer.
4. Train longer and push past 99%.